# 02 — HDD Regression & Normalization

Run OLS regression per postal code, apply quality filters, normalize slopes by building stock characteristics from the tax roll, and produce a thermal intensity metric for ranking.

## 2.1 — Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind')

gas_df = pd.read_csv('data/ces_gas_consumption.csv', parse_dates=['month'])
mpac_df = pd.read_csv('data/ces_mpac_summary.csv')

print(f'Gas data: {gas_df.shape}, MPAC data: {mpac_df.shape}')

## 2.2 — Per-Postal-Code OLS Regression

In [ ]:
results = []
for pc, group in gas_df.groupby('postal_code'):
    hdd = group.hdd.values
    gas = group.gas_gj.values

    slope, intercept, r_value, p_value, std_err = stats.linregress(hdd, gas)

    results.append({
        'postal_code': pc,
        'slope': slope,
        'intercept': intercept,
        'r_squared': r_value**2,
        'p_value': p_value,
        'std_err': std_err,
        'n_obs': len(group),
    })

reg_df = pd.DataFrame(results)
print(f'Regressions complete: {len(reg_df)} postal codes')
reg_df.describe().round(4)

## 2.3 — Regression Quality Checks

In [ ]:
# Quality filters
reg_df['pass_r2'] = reg_df.r_squared > 0.80
reg_df['pass_slope'] = reg_df.slope > 0
reg_df['pass_intercept'] = reg_df.intercept > 0
reg_df['pass_all'] = reg_df.pass_r2 & reg_df.pass_slope & reg_df.pass_intercept

n_pass = reg_df.pass_all.sum()
n_fail = (~reg_df.pass_all).sum()
print(f'Quality check: {n_pass} pass, {n_fail} flagged')

# R² distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(reg_df.r_squared, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(0.80, color='red', linestyle='--', label='R² threshold = 0.80')
ax.set_xlabel('R²')
ax.set_ylabel('Postal Codes')
ax.set_title('Regression R² Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 2.4 — Merge with MPAC Data and Compute Heated Volume

In [ ]:
# Filter to passing postal codes
qualified = reg_df[reg_df.pass_all].merge(mpac_df, on='postal_code')

# Estimate heated volume per property
FLOOR_HEIGHT = 2.7  # m, above grade
BSMT_HEIGHT = 2.4   # m

qualified['above_grade_vol'] = (qualified.avg_footprint_m2
                                 * qualified.avg_storeys
                                 * FLOOR_HEIGHT)
qualified['below_grade_vol'] = (qualified.avg_footprint_m2
                                 * qualified.basement_fraction
                                 * BSMT_HEIGHT)
qualified['heated_vol_m3'] = qualified.above_grade_vol + qualified.below_grade_vol

print(f'Qualified postal codes: {len(qualified)}')
qualified[['postal_code', 'avg_footprint_m2', 'avg_storeys',
           'basement_fraction', 'heated_vol_m3']].describe().round(1)

## 2.5 — Normalize: Per-Property Slope and Thermal Intensity

In [ ]:
# Per-property slope
qualified['slope_per_prop'] = qualified.slope / qualified.property_count

# Normalized thermal intensity: gas per HDD per m³ heated volume
qualified['thermal_intensity'] = qualified.slope_per_prop / qualified.heated_vol_m3

print('Thermal intensity statistics:')
qualified.thermal_intensity.describe().round(6)

## 2.6 — Effect of Normalization

Compare raw slope ranking vs. normalized ranking — they should differ substantially.

In [ ]:
qualified['rank_raw'] = qualified.slope.rank(ascending=False).astype(int)
qualified['rank_normalized'] = qualified.thermal_intensity.rank(ascending=False).astype(int)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(qualified.rank_raw, qualified.rank_normalized, alpha=0.5, s=20)
ax.plot([0, len(qualified)], [0, len(qualified)], 'r--', alpha=0.5, label='No change')
ax.set_xlabel('Rank by Raw Slope')
ax.set_ylabel('Rank by Normalized Thermal Intensity')
ax.set_title('Normalization Changes the Ranking')
ax.legend()
plt.tight_layout()
plt.show()

# Rank correlation
from scipy.stats import spearmanr
rho, p = spearmanr(qualified.rank_raw, qualified.rank_normalized)
print(f'Spearman rank correlation: {rho:.3f} (p={p:.4f})')
print('Normalization materially reshuffles the ranking.')

## 2.7 — Thermal Intensity by Structure Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for stype in ['detached', 'semi-detached', 'row', 'low-rise-apt']:
    subset = qualified[qualified.structure_type == stype]
    ax.hist(subset.thermal_intensity, bins=20, alpha=0.5, label=stype)

ax.set_xlabel('Normalized Thermal Intensity (GJ / HDD / m³)')
ax.set_ylabel('Postal Codes')
ax.set_title('Thermal Intensity Distribution by Structure Type')
ax.legend()
plt.tight_layout()
plt.show()

## 2.8 — Validate Against Ground Truth

Because this is synthetic data, we can check if the normalized metric correlates with the true slope.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(qualified.true_slope, qualified.thermal_intensity, alpha=0.5, s=20)
ax.set_xlabel('True Thermal Slope (known ground truth)')
ax.set_ylabel('Normalized Thermal Intensity (computed)')
ax.set_title('Validation: Normalized Metric vs. Ground Truth')

rho, p = spearmanr(qualified.true_slope, qualified.thermal_intensity)
ax.text(0.05, 0.95, f'Spearman ρ = {rho:.3f}', transform=ax.transAxes, va='top',
        fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

## 2.9 — Save Results

In [ ]:
qualified.to_csv('data/ces_normalized_results.csv', index=False)
print(f'Saved {len(qualified)} qualified postal codes with thermal intensity.')

---
**Next:** Notebook 03 ranks postal codes, identifies priority neighbourhoods, and builds the targeting output.